In [1]:
import torch

# Exercise 13

Use autograd to find the gradient vector of f(x, y) = sin(x2 y) at the point (x, y) = (1.2, 3.4).

In [2]:
x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)

f = torch.sin(x**2 * y)
f.backward()



In [3]:
x.grad, y.grad

(tensor(1.4899), tensor(0.2629))

# Exercise 15

Build and train a classification MLP on the CoverType dataset:







    

a. Load the dataset using sklearn.datasets.fetch_covtype() and create a custom PyTorch Dataset for this data.

b. Create data loaders for training, validation, and testing.

In [4]:
from sklearn.datasets import fetch_covtype

cov_type = fetch_covtype()
cov_type

{'data': array([[2.596e+03, 5.100e+01, 3.000e+00, ..., 0.000e+00, 0.000e+00,
         0.000e+00],
        [2.590e+03, 5.600e+01, 2.000e+00, ..., 0.000e+00, 0.000e+00,
         0.000e+00],
        [2.804e+03, 1.390e+02, 9.000e+00, ..., 0.000e+00, 0.000e+00,
         0.000e+00],
        ...,
        [2.386e+03, 1.590e+02, 1.700e+01, ..., 0.000e+00, 0.000e+00,
         0.000e+00],
        [2.384e+03, 1.700e+02, 1.500e+01, ..., 0.000e+00, 0.000e+00,
         0.000e+00],
        [2.383e+03, 1.650e+02, 1.300e+01, ..., 0.000e+00, 0.000e+00,
         0.000e+00]], shape=(581012, 54)),
 'target': array([5, 5, 2, ..., 3, 3, 3], shape=(581012,), dtype=int32),
 'frame': None,
 'target_names': ['Cover_Type'],
 'feature_names': ['Elevation',
  'Aspect',
  'Slope',
  'Horizontal_Distance_To_Hydrology',
  'Vertical_Distance_To_Hydrology',
  'Horizontal_Distance_To_Roadways',
  'Hillshade_9am',
  'Hillshade_Noon',
  'Hillshade_3pm',
  'Horizontal_Distance_To_Fire_Points',
  'Wilderness_Area_0',
  'Wilde

In [6]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    cov_type.data, cov_type.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s = scaler.transform(X_test)

In [8]:
X_train_s = torch.FloatTensor(X_train_s)
X_valid_s = torch.FloatTensor(X_valid_s)
X_test_s = torch.FloatTensor(X_test_s)

y_train = (torch.LongTensor(y_train) - 1)  #.reshape(-1, 1)
y_valid = (torch.LongTensor(y_valid) - 1)  #.reshape(-1, 1)
y_test = (torch.LongTensor(y_test) - 1)    #.reshape(-1, 1)

In [10]:
y_train.shape

torch.Size([326819])

In [11]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

train_dataset = TensorDataset(X_train_s, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(X_valid_s, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)

test_dataset = TensorDataset(X_test_s, y_test)
test_loader = DataLoader(test_dataset, batch_size=32)


c. Build a custom MLP module to tackle this classification task. You can optionally use the custom Dense module from the previous exercise.

In [20]:
import torch.nn as nn
import numpy as np


n_features = len(cov_type.feature_names)
n_classes = len(np.unique(cov_type.target))

model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, n_classes)
)

d. Train this model on the GPU, and try to reach 93% accuracy on the test set. For this, you will likely have to perform hyperparameter search to find the right number of layers and neurons per layer, a good learning rate and batch size, and so on. You can optionally use Optuna for this.

In [45]:
def evaluate_tm(model, data_loader, metric, device=None):
    if device:
        model = model.to(device)
        metric = metric.to(device)
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end

def train(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss = total_loss +  loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric, device=device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [22]:
import torchmetrics

learning_rate = 0.2
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
xentropy = nn.CrossEntropyLoss()  # Training loss
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes)  # Evaluation metric


In [23]:
n_epochs = 20
history = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs)
# train(model, optimizer, xentropy, train_loader, valid_loader, accuracy, n_epochs)

Epoch 1/20, train loss: 0.5742, train metric: 0.7542, valid metric: 0.7832
Epoch 2/20, train loss: 2.4948, train metric: 0.7027, valid metric: 0.5133
Epoch 3/20, train loss: 1.0078, train metric: 0.5978, valid metric: 0.5501
Epoch 4/20, train loss: 0.9586, train metric: 0.6161, valid metric: 0.6418
Epoch 5/20, train loss: 0.9273, train metric: 0.6280, valid metric: 0.6434
Epoch 6/20, train loss: 0.8810, train metric: 0.6415, valid metric: 0.6501
Epoch 7/20, train loss: 0.8210, train metric: 0.6606, valid metric: 0.6070
Epoch 8/20, train loss: 0.7970, train metric: 0.6691, valid metric: 0.5489
Epoch 9/20, train loss: 0.7681, train metric: 0.6780, valid metric: 0.5416
Epoch 10/20, train loss: 0.7513, train metric: 0.6838, valid metric: 0.4392
Epoch 11/20, train loss: 0.7365, train metric: 0.6869, valid metric: 0.6886
Epoch 12/20, train loss: 0.7136, train metric: 0.6935, valid metric: 0.6868
Epoch 13/20, train loss: 0.7080, train metric: 0.6966, valid metric: 0.6076
Epoch 14/20, train lo

In [27]:
class CovTypeClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_inputs, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_classes))
        
    def forward(self, X):
        return self.mlp(X)

In [28]:
import optuna
from optuna.trial import Trial

def objective(trial: Trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = CovTypeClassifier(n_inputs=n_features, n_hidden=n_hidden, n_classes=n_classes)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes)
    history = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=10)
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy
    

In [29]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2026-02-01 20:03:50,862] A new study created in memory with name: no-name-91de6df6-a361-4499-be50-6e0fd0635822


Epoch 1/10, train loss: 1.4169, train metric: 0.4787, valid metric: 0.4955
Epoch 2/10, train loss: 1.0797, train metric: 0.5611, valid metric: 0.6194
Epoch 3/10, train loss: 0.8849, train metric: 0.6585, valid metric: 0.6782
Epoch 4/10, train loss: 0.7844, train metric: 0.6887, valid metric: 0.6966
Epoch 5/10, train loss: 0.7368, train metric: 0.7053, valid metric: 0.7102
Epoch 6/10, train loss: 0.7096, train metric: 0.7147, valid metric: 0.7173
Epoch 7/10, train loss: 0.6923, train metric: 0.7209, valid metric: 0.7236
Epoch 8/10, train loss: 0.6803, train metric: 0.7265, valid metric: 0.7284
Epoch 9/10, train loss: 0.6710, train metric: 0.7301, valid metric: 0.7314


[I 2026-02-01 20:05:05,510] Trial 0 finished with value: 0.7331833839416504 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7331833839416504.


Epoch 10/10, train loss: 0.6631, train metric: 0.7321, valid metric: 0.7332
Epoch 1/10, train loss: 0.7233, train metric: 0.7053, valid metric: 0.7452
Epoch 2/10, train loss: 0.5730, train metric: 0.7564, valid metric: 0.7469
Epoch 3/10, train loss: 0.5286, train metric: 0.7742, valid metric: 0.7623
Epoch 4/10, train loss: 0.4938, train metric: 0.7895, valid metric: 0.7962
Epoch 5/10, train loss: 0.4650, train metric: 0.8035, valid metric: 0.7061
Epoch 6/10, train loss: 0.4412, train metric: 0.8141, valid metric: 0.7873
Epoch 7/10, train loss: 0.4218, train metric: 0.8226, valid metric: 0.8154
Epoch 8/10, train loss: 0.4039, train metric: 0.8310, valid metric: 0.8007
Epoch 9/10, train loss: 0.3890, train metric: 0.8374, valid metric: 0.7886


[I 2026-02-01 20:06:10,595] Trial 1 finished with value: 0.8429043292999268 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8429043292999268.


...

Epoch 10/10, train loss: 1.1916, train metric: 0.4877, valid metric: 0.4880
Epoch 1/10, train loss: 1.8895, train metric: 0.3654, valid metric: 0.3672
Epoch 2/10, train loss: 1.8110, train metric: 0.3707, valid metric: 0.3657
Epoch 3/10, train loss: 1.7333, train metric: 0.4000, valid metric: 0.4672
Epoch 4/10, train loss: 1.6550, train metric: 0.4867, valid metric: 0.4881
Epoch 5/10, train loss: 1.5775, train metric: 0.4876, valid metric: 0.4880
Epoch 6/10, train loss: 1.5045, train metric: 0.4876, valid metric: 0.4880
Epoch 7/10, train loss: 1.4407, train metric: 0.4876, valid metric: 0.4880
Epoch 8/10, train loss: 1.3890, train metric: 0.4876, valid metric: 0.4880
Epoch 9/10, train loss: 1.3494, train metric: 0.4876, valid metric: 0.4880


[I 2026-02-01 20:08:11,100] Trial 3 finished with value: 0.48807600140571594 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8429043292999268.


Epoch 10/10, train loss: 1.3192, train metric: 0.4876, valid metric: 0.4880
Epoch 1/10, train loss: 0.8870, train metric: 0.6513, valid metric: 0.7272
Epoch 2/10, train loss: 0.6526, train metric: 0.7349, valid metric: 0.7397
Epoch 3/10, train loss: 0.6145, train metric: 0.7443, valid metric: 0.7483
Epoch 4/10, train loss: 0.5881, train metric: 0.7517, valid metric: 0.7573
Epoch 5/10, train loss: 0.5672, train metric: 0.7604, valid metric: 0.7650
Epoch 6/10, train loss: 0.5490, train metric: 0.7671, valid metric: 0.7719
Epoch 7/10, train loss: 0.5331, train metric: 0.7742, valid metric: 0.7654
Epoch 8/10, train loss: 0.5184, train metric: 0.7802, valid metric: 0.7852
Epoch 9/10, train loss: 0.5040, train metric: 0.7869, valid metric: 0.7851


[I 2026-02-01 20:09:17,595] Trial 4 finished with value: 0.7852671146392822 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8429043292999268.


Epoch 10/10, train loss: 0.4904, train metric: 0.7936, valid metric: 0.7853


In [30]:
study.best_params, study.best_value

({'learning_rate': 0.008471801418819975, 'n_hidden': 188}, 0.8429043292999268)

In [31]:
model = CovTypeClassifier(n_inputs=n_features, n_hidden=study.best_params['n_hidden'], n_classes=n_classes)
learning_rate = study.best_params['learning_rate']
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes)
n_epochs = 20
history = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs)

Epoch 1/20, train loss: 0.7200, train metric: 0.7055, valid metric: 0.7221
Epoch 2/20, train loss: 0.5743, train metric: 0.7568, valid metric: 0.7544
Epoch 3/20, train loss: 0.5304, train metric: 0.7735, valid metric: 0.7753
Epoch 4/20, train loss: 0.4976, train metric: 0.7878, valid metric: 0.7541
Epoch 5/20, train loss: 0.4700, train metric: 0.8003, valid metric: 0.7916
Epoch 6/20, train loss: 0.4463, train metric: 0.8114, valid metric: 0.8159
Epoch 7/20, train loss: 0.4265, train metric: 0.8203, valid metric: 0.7813
Epoch 8/20, train loss: 0.4093, train metric: 0.8274, valid metric: 0.8284
Epoch 9/20, train loss: 0.3929, train metric: 0.8355, valid metric: 0.8402
Epoch 10/20, train loss: 0.3805, train metric: 0.8406, valid metric: 0.8198
Epoch 11/20, train loss: 0.3683, train metric: 0.8470, valid metric: 0.7935
Epoch 12/20, train loss: 0.3574, train metric: 0.8515, valid metric: 0.8507
Epoch 13/20, train loss: 0.3472, train metric: 0.8560, valid metric: 0.8563
Epoch 14/20, train lo

In [32]:
history_cont = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=10)

Epoch 1/10, train loss: 0.2883, train metric: 0.8820, valid metric: 0.7987
Epoch 2/10, train loss: 0.2819, train metric: 0.8845, valid metric: 0.8273
Epoch 3/10, train loss: 0.2763, train metric: 0.8871, valid metric: 0.8738
Epoch 4/10, train loss: 0.2716, train metric: 0.8890, valid metric: 0.8884
Epoch 5/10, train loss: 0.2669, train metric: 0.8911, valid metric: 0.8823
Epoch 6/10, train loss: 0.2617, train metric: 0.8933, valid metric: 0.8839
Epoch 7/10, train loss: 0.2580, train metric: 0.8954, valid metric: 0.8859
Epoch 8/10, train loss: 0.2523, train metric: 0.8975, valid metric: 0.8946
Epoch 9/10, train loss: 0.2484, train metric: 0.8993, valid metric: 0.8994
Epoch 10/10, train loss: 0.2446, train metric: 0.9003, valid metric: 0.8982


In [33]:
evaluate_tm(model, test_loader, accuracy)

tensor(0.8968)

In [35]:
history_cont = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=5)

Epoch 1/5, train loss: 0.2405, train metric: 0.9018, valid metric: 0.7726
Epoch 2/5, train loss: 0.2370, train metric: 0.9034, valid metric: 0.8996
Epoch 3/5, train loss: 0.2338, train metric: 0.9050, valid metric: 0.9047
Epoch 4/5, train loss: 0.2303, train metric: 0.9065, valid metric: 0.8304
Epoch 5/5, train loss: 0.2274, train metric: 0.9077, valid metric: 0.9048


In [36]:
evaluate_tm(model, test_loader, accuracy)

tensor(0.9038)

In [46]:
history_cont = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=5, device="mps")

Epoch 1/5, train loss: 0.2122, train metric: 0.9138, valid metric: 0.8841
Epoch 2/5, train loss: 0.2099, train metric: 0.9149, valid metric: 0.8812
Epoch 3/5, train loss: 0.2076, train metric: 0.9158, valid metric: 0.8812
Epoch 4/5, train loss: 0.2050, train metric: 0.9168, valid metric: 0.9094
Epoch 5/5, train loss: 0.2026, train metric: 0.9182, valid metric: 0.9139


In [48]:
evaluate_tm(model, test_loader, accuracy, device="cpu")

tensor(0.9134)

In [49]:
evaluate_tm(model, test_loader, accuracy, device="mps")

tensor(0.9134, device='mps:0')

In [54]:
test_loader = DataLoader(test_dataset, batch_size=1024)
evaluate_tm(model, test_loader, accuracy, device="mps")


tensor(0.9134, device='mps:0')

In [55]:
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=1024)

history_cont = train(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=5, device="mps")

Epoch 1/5, train loss: 0.1630, train metric: 0.9372, valid metric: 0.9281
Epoch 2/5, train loss: 0.1560, train metric: 0.9406, valid metric: 0.9296
Epoch 3/5, train loss: 0.1542, train metric: 0.9413, valid metric: 0.9301
Epoch 4/5, train loss: 0.1530, train metric: 0.9419, valid metric: 0.9303
Epoch 5/5, train loss: 0.1521, train metric: 0.9424, valid metric: 0.9302


!!! So using a large batch size (1024) does speed up execution on the GPU!